In [1]:
!pip -q install sentence-transformers faiss-cpu openai pandas numpy psutil

In [2]:
import json
import os
import re
import time
from getpass import getpass
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import psutil
from sentence_transformers import SentenceTransformer

DOCUMENTS_JSON = r"""[
  {
    "id": "P-01",
    "title": "Suvarma layihəsi",
    "category": "İnfrastruktur",
    "budget_azn": 90000,
    "duration_months": 4,
    "region": "Şəki",
    "status": "Planlaşdırılır",
    "description": "Kənd təsərrüfatı sahələrinin su təminatını yaxşılaşdırmaq üçün nəzərdə tutulub.",
    "text": "P-01 — Suvarma layihəsi. Kateqoriya: İnfrastruktur. Büdcə: 90 000 AZN. Müddət: 4 ay. Region: Şəki. Status: Planlaşdırılır. Kənd təsərrüfatı sahələrinin su təminatını yaxşılaşdırmaq üçün nəzərdə tutulub."
  },
  {
    "id": "P-02",
    "title": "Kənd yolu layihəsi",
    "category": "İnfrastruktur",
    "budget_azn": 115000,
    "duration_months": 7,
    "region": "Ağcabədi",
    "status": "Davam edir",
    "description": "Kəndlərarası nəqliyyat əlaqəsini yaxşılaşdırmaq üçün yol təmiri aparılır.",
    "text": "P-02 — Kənd yolu layihəsi. Kateqoriya: İnfrastruktur. Büdcə: 115 000 AZN. Müddət: 7 ay. Region: Ağcabədi. Status: Davam edir. Kəndlərarası nəqliyyat əlaqəsini yaxşılaşdırmaq üçün yol təmiri aparılır."
  },
  {
    "id": "P-03",
    "title": "Su anbarı layihəsi",
    "category": "İnfrastruktur",
    "budget_azn": 180000,
    "duration_months": 12,
    "region": "Tovuz",
    "status": "Planlaşdırılır",
    "description": "Su ehtiyatlarının saxlanması və idarə olunması üçün yeni su anbarı yaradılacaq.",
    "text": "P-03 — Su anbarı layihəsi. Kateqoriya: İnfrastruktur. Büdcə: 180 000 AZN. Müddət: 12 ay. Region: Tovuz. Status: Planlaşdırılır. Su ehtiyatlarının saxlanması və idarə olunması üçün yeni su anbarı yaradılacaq."
  },
  {
    "id": "P-04",
    "title": "Şəhər parkı layihəsi",
    "category": "İnfrastruktur",
    "budget_azn": 65000,
    "duration_months": 5,
    "region": "Mingəçevir",
    "status": "Tamamlanıb",
    "description": "Şəhər sakinləri üçün yeni istirahət və yaşıllıq sahəsi yaradılıb.",
    "text": "P-04 — Şəhər parkı layihəsi. Kateqoriya: İnfrastruktur. Büdcə: 65 000 AZN. Müddət: 5 ay. Region: Mingəçevir. Status: Tamamlanıb. Şəhər sakinləri üçün yeni istirahət və yaşıllıq sahəsi yaradılıb."
  },
  {
    "id": "P-05",
    "title": "Körpü yenilənməsi layihəsi",
    "category": "İnfrastruktur",
    "budget_azn": 145000,
    "duration_months": 9,
    "region": "Qusar",
    "status": "Davam edir",
    "description": "Nəqliyyat təhlükəsizliyini artırmaq üçün mövcud körpü yenilənir.",
    "text": "P-05 — Körpü yenilənməsi layihəsi. Kateqoriya: İnfrastruktur. Büdcə: 145 000 AZN. Müddət: 9 ay. Region: Qusar. Status: Davam edir. Nəqliyyat təhlükəsizliyini artırmaq üçün mövcud körpü yenilənir."
  },
  {
    "id": "P-06",
    "title": "Kanalizasiya layihəsi",
    "category": "İnfrastruktur",
    "budget_azn": 210000,
    "duration_months": 14,
    "region": "Lənkəran",
    "status": "Planlaşdırılır",
    "description": "Şəhərin kanalizasiya infrastrukturunun yaxşılaşdırılması nəzərdə tutulur.",
    "text": "P-06 — Kanalizasiya layihəsi. Kateqoriya: İnfrastruktur. Büdcə: 210 000 AZN. Müddət: 14 ay. Region: Lənkəran. Status: Planlaşdırılır. Şəhərin kanalizasiya infrastrukturunun yaxşılaşdırılması nəzərdə tutulur."
  },
  {
    "id": "P-07",
    "title": "Günəş enerjisi layihəsi",
    "category": "Enerji",
    "budget_azn": 240000,
    "duration_months": 15,
    "region": "Naxçıvan",
    "status": "Davam edir",
    "description": "Günəş panelləri vasitəsilə elektrik enerjisi istehsal ediləcək.",
    "text": "P-07 — Günəş enerjisi layihəsi. Kateqoriya: Enerji. Büdcə: 240 000 AZN. Müddət: 15 ay. Region: Naxçıvan. Status: Davam edir. Günəş panelləri vasitəsilə elektrik enerjisi istehsal ediləcək."
  },
  {
    "id": "P-08",
    "title": "Külək enerjisi layihəsi",
    "category": "Enerji",
    "budget_azn": 320000,
    "duration_months": 18,
    "region": "Abşeron",
    "status": "Planlaşdırılır",
    "description": "Külək enerjisindən istifadə etməklə yeni enerji mənbəyi yaradılacaq.",
    "text": "P-08 — Külək enerjisi layihəsi. Kateqoriya: Enerji. Büdcə: 320 000 AZN. Müddət: 18 ay. Region: Abşeron. Status: Planlaşdırılır. Külək enerjisindən istifadə etməklə yeni enerji mənbəyi yaradılacaq."
  },
  {
    "id": "P-09",
    "title": "Enerji səmərəliliyi layihəsi",
    "category": "Enerji",
    "budget_azn": 85000,
    "duration_months": 6,
    "region": "Bakı",
    "status": "Tamamlanıb",
    "description": "İctimai binalarda enerji sərfiyyatının azaldılması üçün tədbirlər görülüb.",
    "text": "P-09 — Enerji səmərəliliyi layihəsi. Kateqoriya: Enerji. Büdcə: 85 000 AZN. Müddət: 6 ay. Region: Bakı. Status: Tamamlanıb. İctimai binalarda enerji sərfiyyatının azaldılması üçün tədbirlər görülüb."
  },
  {
    "id": "P-10",
    "title": "Elektrik şəbəkəsi layihəsi",
    "category": "Enerji",
    "budget_azn": 175000,
    "duration_months": 10,
    "region": "Gəncə",
    "status": "Davam edir",
    "description": "Elektrik paylama şəbəkəsinin etibarlılığını artırmaq üçün yenilənmə aparılır.",
    "text": "P-10 — Elektrik şəbəkəsi layihəsi. Kateqoriya: Enerji. Büdcə: 175 000 AZN. Müddət: 10 ay. Region: Gəncə. Status: Davam edir. Elektrik paylama şəbəkəsinin etibarlılığını artırmaq üçün yenilənmə aparılır."
  },
  {
    "id": "P-11",
    "title": "Batareya saxlama layihəsi",
    "category": "Enerji",
    "budget_azn": 195000,
    "duration_months": 11,
    "region": "Sumqayıt",
    "status": "Planlaşdırılır",
    "description": "Elektrik enerjisinin saxlanması üçün batareya sistemi qurulacaq.",
    "text": "P-11 — Batareya saxlama layihəsi. Kateqoriya: Enerji. Büdcə: 195 000 AZN. Müddət: 11 ay. Region: Sumqayıt. Status: Planlaşdırılır. Elektrik enerjisinin saxlanması üçün batareya sistemi qurulacaq."
  },
  {
    "id": "P-12",
    "title": "Biokütlə enerji layihəsi",
    "category": "Enerji",
    "budget_azn": 155000,
    "duration_months": 8,
    "region": "Salyan",
    "status": "Planlaşdırılır",
    "description": "Kənd təsərrüfatı tullantılarından enerji istehsalı həyata keçiriləcək.",
    "text": "P-12 — Biokütlə enerji layihəsi. Kateqoriya: Enerji. Büdcə: 155 000 AZN. Müddət: 8 ay. Region: Salyan. Status: Planlaşdırılır. Kənd təsərrüfatı tullantılarından enerji istehsalı həyata keçiriləcək."
  },
  {
    "id": "P-13",
    "title": "Məktəb laboratoriyası layihəsi",
    "category": "Təhsil",
    "budget_azn": 75000,
    "duration_months": 6,
    "region": "Quba",
    "status": "Tamamlanıb",
    "description": "Məktəblərdə müasir təcrübə laboratoriyalarının yaradılması nəzərdə tutulub.",
    "text": "P-13 — Məktəb laboratoriyası layihəsi. Kateqoriya: Təhsil. Büdcə: 75 000 AZN. Müddət: 6 ay. Region: Quba. Status: Tamamlanıb. Məktəblərdə müasir təcrübə laboratoriyalarının yaradılması nəzərdə tutulub."
  },
  {
    "id": "P-14",
    "title": "Rəqəmsal sinif layihəsi",
    "category": "Təhsil",
    "budget_azn": 68000,
    "duration_months": 5,
    "region": "Bərdə",
    "status": "Davam edir",
    "description": "Məktəblərin kompüter və rəqəmsal avadanlıqlarla təmin olunması həyata keçirilir.",
    "text": "P-14 — Rəqəmsal sinif layihəsi. Kateqoriya: Təhsil. Büdcə: 68 000 AZN. Müddət: 5 ay. Region: Bərdə. Status: Davam edir. Məktəblərin kompüter və rəqəmsal avadanlıqlarla təmin olunması həyata keçirilir."
  },
  {
    "id": "P-15",
    "title": "Müəllim təlimi layihəsi",
    "category": "Təhsil",
    "budget_azn": 45000,
    "duration_months": 4,
    "region": "Şamaxı",
    "status": "Planlaşdırılır",
    "description": "Müəllimlərin yeni tədris metodları üzrə hazırlanması təşkil ediləcək.",
    "text": "P-15 — Müəllim təlimi layihəsi. Kateqoriya: Təhsil. Büdcə: 45 000 AZN. Müddət: 4 ay. Region: Şamaxı. Status: Planlaşdırılır. Müəllimlərin yeni tədris metodları üzrə hazırlanması təşkil ediləcək."
  },
  {
    "id": "P-16",
    "title": "Kitabxana yenilənməsi layihəsi",
    "category": "Təhsil",
    "budget_azn": 52000,
    "duration_months": 5,
    "region": "Ağdaş",
    "status": "Tamamlanıb",
    "description": "Məktəb kitabxanalarının kitab fondu və texniki imkanları yenilənib.",
    "text": "P-16 — Kitabxana yenilənməsi layihəsi. Kateqoriya: Təhsil. Büdcə: 52 000 AZN. Müddət: 5 ay. Region: Ağdaş. Status: Tamamlanıb. Məktəb kitabxanalarının kitab fondu və texniki imkanları yenilənib."
  },
  {
    "id": "P-17",
    "title": "Tələbə təqaüdü layihəsi",
    "category": "Təhsil",
    "budget_azn": 125000,
    "duration_months": 12,
    "region": "Bakı",
    "status": "Davam edir",
    "description": "Yüksək nəticə göstərən tələbələrə maliyyə dəstəyi verilir.",
    "text": "P-17 — Tələbə təqaüdü layihəsi. Kateqoriya: Təhsil. Büdcə: 125 000 AZN. Müddət: 12 ay. Region: Bakı. Status: Davam edir. Yüksək nəticə göstərən tələbələrə maliyyə dəstəyi verilir."
  },
  {
    "id": "P-18",
    "title": "Peşə hazırlığı layihəsi",
    "category": "Təhsil",
    "budget_azn": 98000,
    "duration_months": 9,
    "region": "Mingəçevir",
    "status": "Planlaşdırılır",
    "description": "Gənclər üçün müxtəlif peşə istiqamətləri üzrə hazırlıq kursları keçiriləcək.",
    "text": "P-18 — Peşə hazırlığı layihəsi. Kateqoriya: Təhsil. Büdcə: 98 000 AZN. Müddət: 9 ay. Region: Mingəçevir. Status: Planlaşdırılır. Gənclər üçün müxtəlif peşə istiqamətləri üzrə hazırlıq kursları keçiriləcək."
  },
  {
    "id": "P-19",
    "title": "Mobil klinika layihəsi",
    "category": "Səhiyyə",
    "budget_azn": 110000,
    "duration_months": 7,
    "region": "Ağsu",
    "status": "Davam edir",
    "description": "Uzaq yaşayış məntəqələrinə tibbi xidmət çatdırılması üçün mobil klinika yaradılır.",
    "text": "P-19 — Mobil klinika layihəsi. Kateqoriya: Səhiyyə. Büdcə: 110 000 AZN. Müddət: 7 ay. Region: Ağsu. Status: Davam edir. Uzaq yaşayış məntəqələrinə tibbi xidmət çatdırılması üçün mobil klinika yaradılır."
  },
  {
    "id": "P-20",
    "title": "Diaqnostika mərkəzi layihəsi",
    "category": "Səhiyyə",
    "budget_azn": 280000,
    "duration_months": 16,
    "region": "Gəncə",
    "status": "Planlaşdırılır",
    "description": "Müasir tibbi diaqnostika xidmətləri göstərən mərkəz yaradılacaq.",
    "text": "P-20 — Diaqnostika mərkəzi layihəsi. Kateqoriya: Səhiyyə. Büdcə: 280 000 AZN. Müddət: 16 ay. Region: Gəncə. Status: Planlaşdırılır. Müasir tibbi diaqnostika xidmətləri göstərən mərkəz yaradılacaq."
  },
  {
    "id": "P-21",
    "title": "Təcili yardım layihəsi",
    "category": "Səhiyyə",
    "budget_azn": 135000,
    "duration_months": 8,
    "region": "Şirvan",
    "status": "Tamamlanıb",
    "description": "Təcili tibbi yardım avtomobillərinin sayı artırılıb.",
    "text": "P-21 — Təcili yardım layihəsi. Kateqoriya: Səhiyyə. Büdcə: 135 000 AZN. Müddət: 8 ay. Region: Şirvan. Status: Tamamlanıb. Təcili tibbi yardım avtomobillərinin sayı artırılıb."
  },
  {
    "id": "P-22",
    "title": "Ana və uşaq sağlamlığı layihəsi",
    "category": "Səhiyyə",
    "budget_azn": 92000,
    "duration_months": 6,
    "region": "Lerik",
    "status": "Davam edir",
    "description": "Ana və uşaqlar üçün profilaktik tibbi xidmətlərin əlçatanlığı artırılır.",
    "text": "P-22 — Ana və uşaq sağlamlığı layihəsi. Kateqoriya: Səhiyyə. Büdcə: 92 000 AZN. Müddət: 6 ay. Region: Lerik. Status: Davam edir. Ana və uşaqlar üçün profilaktik tibbi xidmətlərin əlçatanlığı artırılır."
  },
  {
    "id": "P-23",
    "title": "Telemedisin layihəsi",
    "category": "Səhiyyə",
    "budget_azn": 160000,
    "duration_months": 10,
    "region": "Qazax",
    "status": "Planlaşdırılır",
    "description": "Uzaq bölgələr üçün onlayn tibbi konsultasiya sistemi qurulacaq.",
    "text": "P-23 — Telemedisin layihəsi. Kateqoriya: Səhiyyə. Büdcə: 160 000 AZN. Müddət: 10 ay. Region: Qazax. Status: Planlaşdırılır. Uzaq bölgələr üçün onlayn tibbi konsultasiya sistemi qurulacaq."
  },
  {
    "id": "P-24",
    "title": "Sağlam qidalanma layihəsi",
    "category": "Səhiyyə",
    "budget_azn": 58000,
    "duration_months": 5,
    "region": "Masallı",
    "status": "Tamamlanıb",
    "description": "Əhali arasında sağlam qidalanma vərdişlərinin artırılması üçün maarifləndirmə aparılıb.",
    "text": "P-24 — Sağlam qidalanma layihəsi. Kateqoriya: Səhiyyə. Büdcə: 58 000 AZN. Müddət: 5 ay. Region: Masallı. Status: Tamamlanıb. Əhali arasında sağlam qidalanma vərdişlərinin artırılması üçün maarifləndirmə aparılıb."
  },
  {
    "id": "P-25",
    "title": "İstixana layihəsi",
    "category": "Kənd təsərrüfatı",
    "budget_azn": 145000,
    "duration_months": 9,
    "region": "Sabirabad",
    "status": "Davam edir",
    "description": "Tərəvəz istehsalını artırmaq üçün müasir istixana kompleksi yaradılır.",
    "text": "P-25 — İstixana layihəsi. Kateqoriya: Kənd təsərrüfatı. Büdcə: 145 000 AZN. Müddət: 9 ay. Region: Sabirabad. Status: Davam edir. Tərəvəz istehsalını artırmaq üçün müasir istixana kompleksi yaradılır."
  },
  {
    "id": "P-26",
    "title": "Taxıl istehsalı layihəsi",
    "category": "Kənd təsərrüfatı",
    "budget_azn": 130000,
    "duration_months": 8,
    "region": "Ağcabədi",
    "status": "Planlaşdırılır",
    "description": "Taxıl məhsuldarlığını artırmaq üçün yeni aqrotexniki üsullar tətbiq ediləcək.",
    "text": "P-26 — Taxıl istehsalı layihəsi. Kateqoriya: Kənd təsərrüfatı. Büdcə: 130 000 AZN. Müddət: 8 ay. Region: Ağcabədi. Status: Planlaşdırılır. Taxıl məhsuldarlığını artırmaq üçün yeni aqrotexniki üsullar tətbiq ediləcək."
  },
  {
    "id": "P-27",
    "title": "Arıçılıq dəstəyi layihəsi",
    "category": "Kənd təsərrüfatı",
    "budget_azn": 62000,
    "duration_months": 5,
    "region": "Balakən",
    "status": "Tamamlanıb",
    "description": "Arıçılıq təsərrüfatlarına avadanlıq və texniki dəstək verilib.",
    "text": "P-27 — Arıçılıq dəstəyi layihəsi. Kateqoriya: Kənd təsərrüfatı. Büdcə: 62 000 AZN. Müddət: 5 ay. Region: Balakən. Status: Tamamlanıb. Arıçılıq təsərrüfatlarına avadanlıq və texniki dəstək verilib."
  },
  {
    "id": "P-28",
    "title": "Meyvə bağları layihəsi",
    "category": "Kənd təsərrüfatı",
    "budget_azn": 118000,
    "duration_months": 10,
    "region": "Xaçmaz",
    "status": "Davam edir",
    "description": "Yeni meyvə bağlarının salınması və məhsuldarlığın artırılması həyata keçirilir.",
    "text": "P-28 — Meyvə bağları layihəsi. Kateqoriya: Kənd təsərrüfatı. Büdcə: 118 000 AZN. Müddət: 10 ay. Region: Xaçmaz. Status: Davam edir. Yeni meyvə bağlarının salınması və məhsuldarlığın artırılması həyata keçirilir."
  },
  {
    "id": "P-29",
    "title": "Soyuducu anbar layihəsi",
    "category": "Kənd təsərrüfatı",
    "budget_azn": 205000,
    "duration_months": 13,
    "region": "Göyçay",
    "status": "Planlaşdırılır",
    "description": "Kənd təsərrüfatı məhsullarının saxlanması üçün soyuducu anbar tikiləcək.",
    "text": "P-29 — Soyuducu anbar layihəsi. Kateqoriya: Kənd təsərrüfatı. Büdcə: 205 000 AZN. Müddət: 13 ay. Region: Göyçay. Status: Planlaşdırılır. Kənd təsərrüfatı məhsullarının saxlanması üçün soyuducu anbar tikiləcək."
  },
  {
    "id": "P-30",
    "title": "Aqrotexnika layihəsi",
    "category": "Kənd təsərrüfatı",
    "budget_azn": 175000,
    "duration_months": 11,
    "region": "Saatlı",
    "status": "Davam edir",
    "description": "Fermerlərə müasir kənd təsərrüfatı texnikasının istifadəsi üçün dəstək göstərilir.",
    "text": "P-30 — Aqrotexnika layihəsi. Kateqoriya: Kənd təsərrüfatı. Büdcə: 175 000 AZN. Müddət: 11 ay. Region: Saatlı. Status: Davam edir. Fermerlərə müasir kənd təsərrüfatı texnikasının istifadəsi üçün dəstək göstərilir."
  }
]"""
documents = json.loads(DOCUMENTS_JSON)
assert len(documents) == 30
assert all({"id", "title", "category", "budget_azn", "duration_months", "region", "status", "description", "text"} <= set(d) for d in documents)

print(f"Sənəd sayı: {len(documents)}")
print(documents[0]["text"])


Sənəd sayı: 30
P-01 — Suvarma layihəsi. Kateqoriya: İnfrastruktur. Büdcə: 90 000 AZN. Müddət: 4 ay. Region: Şəki. Status: Planlaşdırılır. Kənd təsərrüfatı sahələrinin su təminatını yaxşılaşdırmaq üçün nəzərdə tutulub.


In [3]:
def chunk_text(text, chunk_size=80, overlap=15):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    step = chunk_size - overlap
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), step) if words[i:i + chunk_size]]

def make_chunks(docs):
    chunks = []
    for doc in docs:
        for number, text in enumerate(chunk_text(doc["text"]), start=1):
            chunks.append({
                "chunk_id": f"{doc['id']}-C{number:02d}",
                "source_id": doc["id"],
                "text": text,
                "document": doc
            })
    return chunks

chunks = make_chunks(documents)
print(f"Chunk sayı: {len(chunks)}")
print(chunks[0]["chunk_id"], chunks[0]["source_id"])


Chunk sayı: 30
P-01-C01 P-01


In [4]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Hugging Face token: ")
login(token=hf_token)
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_model = SentenceTransformer(embedding_model_name)
chunk_vectors = embedding_model.encode(
    [chunk["text"] for chunk in chunks],
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

faiss_index = faiss.IndexFlatIP(chunk_vectors.shape[1])
faiss_index.add(chunk_vectors)

def retrieve(question, top_k=3):
    question_upper = question.upper()

    requested_ids = list(
        dict.fromkeys(
            re.findall(r"\bP-\d+\b", question_upper)
        )
    )

    direct_hits = [
        {
            "chunk": chunk,
            "score": 1.0
        }
        for chunk in chunks
        if chunk["source_id"].upper() in requested_ids
    ]

    if direct_hits:
        return direct_hits[:top_k]

    query_vector = embedding_model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    search_k = min(max(top_k * 5, top_k), len(chunks))

    scores, positions = faiss_index.search(
        query_vector,
        search_k
    )

    results = []
    seen_sources = set()

    for score, position in zip(scores[0], positions[0]):
        position = int(position)

        if position < 0:
            continue

        chunk = chunks[position]
        source_id = chunk["source_id"]

        if source_id in seen_sources:
            continue

        results.append({
            "chunk": chunk,
            "score": float(score)
        })

        seen_sources.add(source_id)

        if len(results) == top_k:
            break

    return results


retrieval_test = retrieve(
    "Enerji layihəsinin büdcəsi nə qədərdir?"
)

for item in retrieval_test:
    print(
        item["chunk"]["source_id"],
        round(item["score"], 4),
        item["chunk"]["document"]["title"]
    )


Hugging Face token: ··········


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

P-08 0.6345 Külək enerjisi layihəsi
P-07 0.6169 Günəş enerjisi layihəsi
P-12 0.5664 Biokütlə enerji layihəsi


In [5]:
SYSTEM_RULES = """Sən Azərbaycan dilində cavab verən mənbəyə əsaslanan köməkçisən.
Yalnız verilmiş KONTEKST-dən istifadə et.
Kontekstdə olmayan faktı uydurma.
Hesablamaları düzgün apar.
Cavabın sonunda istifadə etdiyin mənbə ID-lərini yaz.
Məlumat yoxdursa, bunu açıq bildir: Bu məlumat təqdim olunan mənbələrdə göstərilməyib.
"""

def build_prompt(question, hits):
    context = "\n".join(
        f"[{hit['chunk']['source_id']}] {hit['chunk']['text']}"
        for hit in hits
    )
    return f"{SYSTEM_RULES}\nKONTEKST:\n{context}\n\nSUAL: {question}\nCAVAB:"

def make_client(base_url, api_key):
    from openai import OpenAI
    return OpenAI(base_url=base_url, api_key=api_key)

def ask_llm(client, model_name, question, top_k=3):
    hits = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, hits)
    started = time.perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_RULES},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=500
    )
    seconds = time.perf_counter() - started

    raw_content = response.choices[0].message.content
    answer = raw_content.strip() if raw_content is not None else "[XƏTA: Model boş cavab qaytardı (None)]"

    return {
        "answer": answer,
        "sources": ", ".join(hit["chunk"]["source_id"] for hit in hits),
        "seconds": round(seconds, 3),
        "hits": hits
    }

In [6]:
def extract_visible_text(message):
    content = getattr(message, "content", None)
    if isinstance(content, str) and content.strip():
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and isinstance(item.get("text"), str):
                parts.append(item["text"])
            elif hasattr(item, "text") and isinstance(item.text, str):
                parts.append(item.text)
        text = "".join(parts).strip()
        if text:
            return text
    return None

def response_diagnostics(response):
    choice = response.choices[0] if getattr(response, "choices", None) else None
    message = getattr(choice, "message", None)
    usage = getattr(response, "usage", None)
    return {
        "finish_reason": getattr(choice, "finish_reason", None),
        "has_reasoning": bool(getattr(message, "reasoning", None)),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "reasoning_tokens": getattr(getattr(usage, "completion_tokens_details", None), "reasoning_tokens", None)
    }

def ask_llm(client, model_name, question, top_k=3):
    hits = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, hits)
    attempts = [
        {"reasoning": {"max_tokens": 256, "exclude": True}},
        {"reasoning": {"effort": "none"}},
        {}
    ]
    errors = []
    started = time.perf_counter()
    for extra_body in attempts:
        try:
            request = {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": SYSTEM_RULES},
                    {"role": "user", "content": prompt + "\n\nYalnız yekun cavabı yaz; boş cavab qaytarma."}
                ],
                "temperature": 0,
                "max_tokens": 1500
            }
            if extra_body:
                request["extra_body"] = extra_body
            response = client.chat.completions.create(**request)
            message = response.choices[0].message
            answer = extract_visible_text(message)
            if answer:
                return {
                    "answer": answer,
                    "sources": ", ".join(hit["chunk"]["source_id"] for hit in hits),
                    "seconds": round(time.perf_counter() - started, 3),
                    "hits": hits
                }
            errors.append(response_diagnostics(response))
        except Exception as exc:
            errors.append({"error": str(exc)})
    raise RuntimeError(f"Model boş cavab qaytardı və retry uğursuz oldu: {errors}")


In [7]:
USE_LLM = True

if USE_LLM:
    base_url = input("OpenAI-compatible BASE_URL: ").strip()
    api_key = getpass("API key: ")
    model_name = input("Model adı: ").strip()
    llm_client = make_client(base_url, api_key)
    question = input("Sual: ").strip()
    result = ask_llm(llm_client, model_name, question)
    print(result["answer"])
    print("Mənbələr:", result["sources"])
else:
    question = "Enerji ilə Suvarma layihəsinin büdcə fərqi nə qədərdir?"
    for item in retrieve(question):
        print(item["chunk"]["source_id"], item["chunk"]["document"]["title"])


OpenAI-compatible BASE_URL: https://openrouter.ai/api/v1
API key: ··········
Model adı: google/gemma-3-4b-it
Sual: Günəş enerjisi layihəsi hansı regiondadır?
Naxçıvan.

[P-07]
Mənbələr: P-07, P-08, P-11


In [8]:
questions = [
    {"id": "Q01", "type": "fact", "question": "P-01 layihəsinin büdcəsi nə qədərdir?"},
    {"id": "Q02", "type": "fact", "question": "Günəş enerjisi layihəsi hansı regiondadır?"},
    {"id": "Q03", "type": "fact", "question": "Diaqnostika mərkəzi layihəsinin müddəti neçə aydır?"},
    {"id": "Q04", "type": "fact", "question": "P-03 hansı kateqoriyaya aiddir?"},
    {"id": "Q05", "type": "fact", "question": "Soyuducu anbar layihəsinin statusu nədir?"},
    {"id": "Q06", "type": "calculation", "question": "Enerji səmərəliliyi layihəsi ilə Suvarma layihəsinin büdcə fərqi nə qədərdir?"},
    {"id": "Q07", "type": "calculation", "question": "Ən yüksək büdcəli layihə hansıdır?"},
    {"id": "Q08", "type": "calculation", "question": "P-01 və P-03 layihələrinin ümumi büdcəsi nə qədərdir?"},
    {"id": "Q09", "type": "calculation", "question": "10 aydan uzun müddətli layihələrdən üçünü yaz."},
    {"id": "Q10", "type": "calculation", "question": "Ən uzunmüddətli layihə neçə ay davam edir?"},
    {"id": "Q11", "type": "unknown", "question": "P-01 layihəsinin rəhbəri kimdir?"},
    {"id": "Q12", "type": "unknown", "question": "Günəş enerjisi layihəsini hansı şirkət maliyyələşdirir?"},
    {"id": "Q13", "type": "unknown", "question": "Layihələrdə neçə işçi çalışır?"},
    {"id": "Q14", "type": "unknown", "question": "P-02 layihəsi hansı tarixdə tamamlanacaq?"},
    {"id": "Q15", "type": "unknown", "question": "Layihələrin gəlirliliyi neçə faizdir?"}
]

assert len(questions) == 15
assert {item["type"] for item in questions} == {"fact", "calculation", "unknown"}
print(pd.Series([item["type"] for item in questions]).value_counts())


fact           5
calculation    5
unknown        5
Name: count, dtype: int64


In [9]:
RUN_COMPARISON = True

if RUN_COMPARISON:
    base_url = input("OpenAI-compatible BASE_URL: ").strip()
    api_key = getpass("API key: ").strip()
    model_a = input("Model A adı: ").strip()
    model_b = input("Model B adı: ").strip()
    client = make_client(base_url, api_key)

    expected = {
        "Q01": {"answer": "90 000 AZN", "sources": {"P-01"}},
        "Q02": {"answer": "Naxçıvan", "sources": {"P-07"}},
        "Q03": {"answer": "16 ay", "sources": {"P-20"}},
        "Q04": {"answer": "İnfrastruktur", "sources": {"P-03"}},
        "Q05": {"answer": "Planlaşdırılır", "sources": {"P-29"}},
        "Q06": {"answer": "5 000 AZN", "sources": {"P-01", "P-09"}},
        "Q07": {"answer": "Külək enerjisi layihəsi — 320 000 AZN", "sources": {"P-08"}},
        "Q08": {"answer": "270 000 AZN", "sources": {"P-01", "P-03"}},
        "Q09": {"answer": "P-03, P-06, P-07", "sources": {"P-03", "P-06", "P-07"}},
        "Q10": {"answer": "18 ay", "sources": {"P-08"}},
        "Q11": {"answer": "Bu məlumat təqdim olunan mənbələrdə göstərilməyib.", "sources": set()},
        "Q12": {"answer": "Bu məlumat təqdim olunan mənbələrdə göstərilməyib.", "sources": set()},
        "Q13": {"answer": "Bu məlumat təqdim olunan mənbələrdə göstərilməyib.", "sources": set()},
        "Q14": {"answer": "Bu məlumat təqdim olunan mənbələrdə göstərilməyib.", "sources": set()},
        "Q15": {"answer": "Bu məlumat təqdim olunan mənbələrdə göstərilməyib.", "sources": set()}
    }

    refusal_phrases = [
        "bu məlumat təqdim olunan mənbələrdə göstərilməyib",
        "mənbələrdə göstərilməyib",
        "məlumat mənbədə yoxdur",
        "məlumat verilməyib",
        "məlum deyil",
        "göstərilməyib"
    ]
    error_phrases = ["[xəta]", "error", "none", "boş cavab", "model boş cavab"]

    def normalize_text(value):
        value = "" if pd.isna(value) else str(value).lower()
        replacements = str.maketrans({"ə": "e", "ı": "i", "ö": "o", "ü": "u", "ş": "s", "ç": "c"})
        value = value.translate(replacements)
        return re.sub(r"[^a-z0-9\s]", " ", value)

    def text_tokens(value):
        stopwords = {"və", "ilə", "olan", "üçün", "layihəsi", "layihə", "nə", "qədər", "hansı"}
        return {
            token for token in normalize_text(value).split()
            if len(token) > 1 and token not in stopwords
        }

    def numbers(value):
        return set(re.findall(r"\d+(?:[,.]\d+)?", normalize_text(value).replace(" ", "")))

    def ids(value):
        return set(re.findall(r"P-\d+", str(value).upper()))

    def has_error(answer):
        text = "" if pd.isna(answer) else str(answer).strip().lower()
        return not text or any(item in text for item in error_phrases)

    def has_refusal(answer):
        text = normalize_text(answer)
        return any(normalize_text(item) in text for item in refusal_phrases)

    def matches(answer, expected_answer):
        answer_text = normalize_text(answer)
        expected_text = normalize_text(expected_answer)
        expected_numbers = numbers(expected_answer)
        answer_numbers = numbers(answer)
        if expected_numbers and not expected_numbers.issubset(answer_numbers):
            return False
        if expected_text in answer_text:
            return True
        expected_words = text_tokens(expected_answer)
        if not expected_words:
            return False
        overlap = len(expected_words & text_tokens(answer)) / len(expected_words)
        if overlap >= 0.75:
            return True
        if overlap >= 0.40:
            return None
        return False

    def language_score(answer):
        if has_error(answer):
            return 0
        text = str(answer).strip()
        if len(text.split()) < 2:
            return 3
        return 5 if any(char in text.lower() for char in "əğıöşüç") else 4

    def score_result(row):
        question_id = row["question_id"]
        question_type = row["type"]
        answer = row["answer"]
        answer_key = expected[question_id]
        error = has_error(answer)
        refusal = has_refusal(answer)
        answer_match = matches(answer, answer_key["answer"])
        source_match = bool(ids(row["sources"]) & answer_key["sources"])

        if error:
            reasoning = 0
            faithfulness = 0
            decision = "error_or_empty_answer"
        elif question_type == "unknown":
            if refusal:
                reasoning = 5
                faithfulness = 5
                decision = "correct_refusal"
            else:
                reasoning = 1
                faithfulness = 1
                decision = "unknown_question_not_refused"
        elif answer_match is True:
            reasoning = 5
            faithfulness = 5 if source_match else 3
            decision = "correct_answer"
        elif answer_match is None:
            reasoning = 3
            faithfulness = 3 if source_match else 2
            decision = "partial_answer"
        else:
            reasoning = 1
            faithfulness = 1 if source_match else 0
            decision = "incorrect_answer"

        manual_review = error or answer_match is not True or (question_type == "unknown" and not refusal)
        notes = (
            f"{decision}; gözlənilən cavab: {answer_key['answer']}; "
            f"gözlənilən mənbələr: {sorted(answer_key['sources'])}; "
            f"tapılan mənbələr: {sorted(ids(row['sources']))}"
        )
        return pd.Series({
            "azerbaijani_0_5": language_score(answer),
            "reasoning_0_5": reasoning,
            "faithfulness_0_5": faithfulness,
            "auto_decision": decision,
            "needs_manual_review": manual_review,
            "notes": notes
        })

    rows = []
    for model_name in [model_a, model_b]:
        for item in questions:
            try:
                result = ask_llm(client, model_name, item["question"], top_k=3)
            except Exception as exc:
                result = {
                    "answer": f"[XƏTA] {exc}",
                    "sources": "",
                    "seconds": None
                }
            rows.append({
                "model": model_name,
                "question_id": item["id"],
                "type": item["type"],
                "question": item["question"],
                "answer": result["answer"],
                "sources": result["sources"],
                "seconds": result["seconds"]
            })

    results = pd.DataFrame(rows)
    scores = results.apply(score_result, axis=1)
    results = pd.concat([results, scores], axis=1)
    score_columns = ["azerbaijani_0_5", "reasoning_0_5", "faithfulness_0_5"]
    results["total_score"] = results[score_columns].sum(axis=1)
    results["average_score"] = results[score_columns].mean(axis=1).round(2)

    results.to_csv("comparison_results_scored.csv", index=False, encoding="utf-8-sig")

    model_summary = (
        results.groupby("model")
        .agg(
            question_count=("question_id", "count"),
            average_azerbaijani=("azerbaijani_0_5", "mean"),
            average_reasoning=("reasoning_0_5", "mean"),
            average_faithfulness=("faithfulness_0_5", "mean"),
            average_total_score=("average_score", "mean"),
            average_response_seconds=("seconds", "mean"),
            manual_review_count=("needs_manual_review", "sum")
        )
        .round(2)
        .reset_index()
    )
    model_summary.to_csv("model_summary.csv", index=False, encoding="utf-8-sig")

    category_summary = (
        results.groupby(["model", "type"])
        .agg(
            question_count=("question_id", "count"),
            average_score=("average_score", "mean"),
            average_reasoning=("reasoning_0_5", "mean"),
            average_faithfulness=("faithfulness_0_5", "mean"),
            average_response_seconds=("seconds", "mean"),
            manual_review_count=("needs_manual_review", "sum")
        )
        .round(2)
        .reset_index()
    )
    category_summary.to_csv("category_summary.csv", index=False, encoding="utf-8-sig")

    manual_review = results[results["needs_manual_review"]]
    manual_review.to_csv("manual_review_required.csv", index=False, encoding="utf-8-sig")

    display(results)
    display(model_summary)
    display(category_summary)
    print(f"comparison_results_scored.csv yaradıldı: {len(results)} sətir")
    print(f"Manual yoxlama tələb edən sətirlər: {len(manual_review)}")

    from google.colab import files
    files.download("comparison_results_scored.csv")
    files.download("model_summary.csv")
    files.download("category_summary.csv")
    files.download("manual_review_required.csv")
else:
    print("Müqayisəni başlatmaq üçün RUN_COMPARISON = True edin.")

OpenAI-compatible BASE_URL: https://openrouter.ai/api/v1
API key: ··········
Model A adı: qwen/qwen3.5-9b
Model B adı: google/gemma-3-4b-it


,model,question_id,type,question,answer,sources,seconds,azerbaijani_0_5,reasoning_0_5,faithfulness_0_5,auto_decision,needs_manual_review,notes,total_score,average_score
0,qwen/qwen3.5-9b,Q01,fact,P-01 layihəsinin büdcəsi nə qədərdir?,P-01 layihəsinin büdcəsi 90 000 AZN-dir.\n[P-01],P-01,8.564,5,5,5,correct_answer,False,correct_answer; gözlənilən cavab: 90 000 AZN; ...,15,5.00
1,qwen/qwen3.5-9b,Q02,fact,Günəş enerjisi layihəsi hansı regiondadır?,Günəş enerjisi layihəsi Naxçıvan regionundadır...,"P-07, P-08, P-11",14.162,5,5,5,correct_answer,False,correct_answer; gözlənilən cavab: Naxçıvan; gö...,15,5.00
2,qwen/qwen3.5-9b,Q03,fact,Diaqnostika mərkəzi layihəsinin müddəti neçə a...,Diaqnostika mərkəzi layihəsinin müddəti 16 ayd...,"P-20, P-29, P-27",17.213,5,5,5,correct_answer,False,correct_answer; gözlənilən cavab: 16 ay; gözlə...,15,5.00
3,qwen/qwen3.5-9b,Q04,fact,P-03 hansı kateqoriyaya aiddir?,"P-03 layihəsi ""İnfrastruktur"" kateqoriyasına a...",P-03,15.672,5,5,5,correct_answer,False,correct_answer; gözlənilən cavab: İnfrastruktu...,15,5.00
4,qwen/qwen3.5-9b,Q05,fact,Soyuducu anbar layihəsinin statusu nədir?,"Soyuducu anbar layihəsinin statusu ""Planlaşdır...","P-28, P-29, P-27",12.604,5,5,5,correct_answer,False,correct_answer; gözlənilən cavab: Planlaşdırıl...,15,5.00
5,qwen/qwen3.5-9b,Q06,calculation,Enerji səmərəliliyi layihəsi ilə Suvarma layih...,"Təqdim olunan mənbələrdə ""Suvarma layihəsi"" ha...","P-09, P-11, P-07",13.181,5,1,1,incorrect_answer,True,incorrect_answer; gözlənilən cavab: 5 000 AZN;...,7,2.33
6,qwen/qwen3.5-9b,Q07,calculation,Ən yüksək büdcəli layihə hansıdır?,Ən yüksək büdcəli layihə P-29 — Soyuducu anbar...,"P-01, P-26, P-29",14.342,5,1,0,incorrect_answer,True,incorrect_answer; gözlənilən cavab: Külək ener...,6,2.00
7,qwen/qwen3.5-9b,Q08,calculation,P-01 və P-03 layihələrinin ümumi büdcəsi nə qə...,"P-01 layihəsinin büdcəsi 90 000 AZN, P-03 layi...","P-01, P-03",13.268,5,5,5,correct_answer,False,correct_answer; gözlənilən cavab: 270 000 AZN;...,15,5.00
8,qwen/qwen3.5-9b,Q09,calculation,10 aydan uzun müddətli layihələrdən üçünü yaz.,"P-26, P-30 və P-15 layihələri 10 aydan uzun mü...","P-26, P-15, P-30",13.300,5,1,0,incorrect_answer,True,"incorrect_answer; gözlənilən cavab: P-03, P-06...",6,2.00
9,qwen/qwen3.5-9b,Q10,calculation,Ən uzunmüddətli layihə neçə ay davam edir?,Ən uzunmüddətli layihə P-29 layihəsidir və 13 ...,"P-27, P-29, P-26",13.470,5,1,0,incorrect_answer,True,incorrect_answer; gözlənilən cavab: 18 ay; göz...,6,2.00


,model,question_count,average_azerbaijani,average_reasoning,average_faithfulness,average_total_score,average_response_seconds,manual_review_count
0,google/gemma-3-4b-it,15,4.53,3.80,3.60,3.98,0.51,5
1,qwen/qwen3.5-9b,15,5.00,3.93,3.73,4.22,14.69,6


,model,type,question_count,average_score,average_reasoning,average_faithfulness,average_response_seconds,manual_review_count
0,google/gemma-3-4b-it,calculation,5,2.53,1.8,1.2,0.57,4
1,google/gemma-3-4b-it,fact,5,4.40,4.6,4.6,0.41,1
2,google/gemma-3-4b-it,unknown,5,5.00,5.0,5.0,0.54,0
3,qwen/qwen3.5-9b,calculation,5,2.67,1.8,1.2,13.51,4
4,qwen/qwen3.5-9b,fact,5,5.00,5.0,5.0,13.64,0
5,qwen/qwen3.5-9b,unknown,5,5.00,5.0,5.0,16.90,2


comparison_results_scored.csv yaradıldı: 30 sətir
Manual yoxlama tələb edən sətirlər: 11


<IPython.core.display.Javascript object>

... (3 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
def system_metrics():
    metrics = {
        "ram_used_gb": round(psutil.virtual_memory().used / (1024 ** 3), 2),
        "ram_total_gb": round(psutil.virtual_memory().total / (1024 ** 3), 2)
    }
    try:
        import torch
        metrics["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU yoxdur"
        metrics["gpu_memory_gb"] = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2) if torch.cuda.is_available() else 0
    except Exception:
        metrics["gpu"] = "torch quraşdırılmayıb"
        metrics["gpu_memory_gb"] = None
    return metrics

print(system_metrics())
if RUN_COMPARISON and not results.empty:
    print(results.groupby("model")["seconds"].agg(["count", "mean"]))


{'ram_used_gb': 4.93, 'ram_total_gb': 12.67, 'gpu': 'GPU yoxdur', 'gpu_memory_gb': 0}
                      count     mean
model                               
google/gemma-3-4b-it     15   0.5074
qwen/qwen3.5-9b          15  14.6854


In [11]:
assert len(documents) >= 30
assert len(chunks) >= len(documents)
assert len(questions) == 15
assert sum(item["type"] == "fact" for item in questions) == 5
assert sum(item["type"] == "calculation" for item in questions) == 5
assert sum(item["type"] == "unknown" for item in questions) == 5
print("Tələblər üzrə statik yoxlama uğurla tamamlandı.")


Tələblər üzrə statik yoxlama uğurla tamamlandı.
